# 03 · Embeddings: Converting Text into Vectors

An **embedding** is a list of numbers (a vector) that represents the **meaning** of a text.
Texts with similar meaning produce vectors that are **close to each other**, usually measured with cosine similarity. This is what makes *semantic search* possible: matching by meaning, not just by keywords.

> RAG pipeline: Load ➜ Split ➜ **Embed** ➜ Store ➜ Retrieve

### Two methods on every LangChain embedding model
| Method | Input | Output | Used for |
|---|---|---|---|
| `embed_query(text)` | one string | `list[float]` | the user's question |
| `embed_documents(texts)` | list of strings | `list[list[float]]` | the document chunks |

### What this notebook covers
1. **OpenAI** embeddings: paid API, needs `OPENAI_API_KEY`
2. Storing embeddings in a **Chroma** vector store + similarity search
3. **Ollama** embeddings: free, run locally
4. **Hugging Face** embeddings: free, run locally with `sentence-transformers`

**Requirements:** a `.env` file with `OPENAI_API_KEY` (and optionally `HF_TOKEN`). See `.env.example` in the project root.
For Ollama: install [Ollama](https://ollama.com) and run `ollama pull nomic-embed-text`.

converting the text into the vectors

## 1. OpenAI Embeddings

### Load the API keys from `.env`
`load_dotenv()` reads the `.env` file and puts its values into environment variables. It returns `True` if a `.env` file was found.

> 🔒 Never hard-code API keys in a notebook. `.env` is listed in `.gitignore`, so it is never pushed to GitHub.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

Make sure the key is set in `os.environ`. `OpenAIEmbeddings` reads `OPENAI_API_KEY` from there automatically.
*(If the key is missing from `.env`, `os.getenv` returns `None` and this line raises a `TypeError`.)*

In [2]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

Create the embedding model. `text-embedding-3-small` returns **1536-dimensional** vectors by default (`text-embedding-3-large` returns 3072).

In [8]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")
embeddings


OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001D413E8FAD0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001D413FEA250>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

Embed a single sentence with `embed_query`:

In [9]:
text = "I am Yokeswaran AI Engineer"
query_result = embeddings.embed_query(text)
query_result 

[-0.0056304931640625,
 0.0226287841796875,
 -0.0160980224609375,
 -0.003993988037109375,
 0.003368377685546875,
 -0.00641632080078125,
 0.018890380859375,
 -0.0262298583984375,
 -0.07275390625,
 -0.00972747802734375,
 0.01763916015625,
 -0.08746337890625,
 -0.0338134765625,
 -0.01328277587890625,
 -0.0030231475830078125,
 0.00969696044921875,
 -0.0230560302734375,
 0.0219573974609375,
 -0.004962921142578125,
 -0.0106048583984375,
 0.02294921875,
 0.0183868408203125,
 -0.0058746337890625,
 0.004100799560546875,
 0.03192138671875,
 -0.00942230224609375,
 0.01354217529296875,
 0.0026798248291015625,
 -0.00827789306640625,
 -0.043853759765625,
 0.0025177001953125,
 -0.037078857421875,
 -0.050628662109375,
 -0.006824493408203125,
 0.0195465087890625,
 -0.0037822723388671875,
 0.0300445556640625,
 0.003856658935546875,
 -0.0157470703125,
 0.003101348876953125,
 -0.005870819091796875,
 -0.0361328125,
 -0.005588531494140625,
 0.039031982421875,
 -0.01351165771484375,
 0.02545166015625,
 -0.036

The length of the vector is the **dimension** of the model: 1536.

In [10]:
len(query_result)

1536

### Custom dimensions
The `text-embedding-3-*` models accept a `dimensions` parameter that returns **shorter vectors**. These are cheaper to store and faster to search, with a small loss in accuracy.

In [11]:
embeddings_720 = OpenAIEmbeddings(model="text-embedding-3-small",dimensions = 720)

In [14]:
text = "I am Yokeswaran AI Engineer"
query_result = embeddings_720.embed_query(text)
query_result 

[-0.007419586181640625,
 0.0298004150390625,
 -0.021209716796875,
 -0.005260467529296875,
 0.004436492919921875,
 -0.008453369140625,
 0.0248870849609375,
 -0.0345458984375,
 -0.0958251953125,
 -0.0128173828125,
 0.0232391357421875,
 -0.11517333984375,
 -0.044525146484375,
 -0.017486572265625,
 -0.0039825439453125,
 0.01276397705078125,
 -0.030364990234375,
 0.0289154052734375,
 -0.00653839111328125,
 -0.0139617919921875,
 0.0302276611328125,
 0.0242156982421875,
 -0.007740020751953125,
 0.005401611328125,
 0.04205322265625,
 -0.01241302490234375,
 0.0178375244140625,
 0.0035305023193359375,
 -0.01090240478515625,
 -0.0577392578125,
 0.0033168792724609375,
 -0.048828125,
 -0.06671142578125,
 -0.0089874267578125,
 0.0257415771484375,
 -0.00498199462890625,
 0.03955078125,
 0.0050811767578125,
 -0.020751953125,
 0.004085540771484375,
 -0.007732391357421875,
 -0.047576904296875,
 -0.00736236572265625,
 0.0513916015625,
 -0.017791748046875,
 0.03350830078125,
 -0.048309326171875,
 -0.01494

Now the vector has 720 dimensions.

In [15]:
len(query_result)

720

## 2. From documents to a Vector Store (Chroma)

A mini pipeline that puts the previous notebooks together: **load** `speech.txt` ➜ **split** into chunks of 500 characters.

In [23]:

from langchain_community.document_loaders import TextLoader

# Step 1: Load the text file
loader = TextLoader("speech.txt")

# Step 2: Load documents
documents = loader.load()

# Step 3: Import text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Step 4: Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Step 5: Split documents
final_documents = text_splitter.split_documents(documents)

# Step 6: Display results
final_documents


[Document(metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence: Concepts and Applications\n\nGood morning everyone!\n\nToday, I am going to talk about Artificial Intelligence, commonly known as AI, and some of its important concepts.\n\nArtificial Intelligence is a branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem-solving, understanding language, and recognizing images.'),
 Document(metadata={'source': 'speech.txt'}, page_content='One of the most important concepts in AI is Machine Learning. Machine Learning is a technique that allows computers to learn from data without being explicitly programmed for every task. For example, email spam filters use machine learning to identify unwanted emails based on previously observed patterns.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Another important concept is Deep

`Chroma.from_documents(docs, embeddings)` embeds every chunk with the given model and stores **vector + text + metadata** in a Chroma vector database.

- Here the DB lives **in memory**. Pass `persist_directory="chroma_db"` to save it to disk.
- ⚠️ `langchain_community.vectorstores.Chroma` is deprecated. The recommended import is `from langchain_chroma import Chroma` (package `langchain-chroma`).

In [24]:
# Vector Embedding and Vector storeDB

from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(final_documents,embeddings)
db

`similarity_search(query)` embeds the query with the **same** embedding model and returns the `k=4` most similar chunks.

In [25]:
query = "Artificial Intelligence is a branch of computer science that focuses on creating machines"
retrieved_results = db.similarity_search(query)
print(retrieved_results)

[Document(metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence: Concepts and Applications\n\nGood morning everyone!\n\nToday, I am going to talk about Artificial Intelligence, commonly known as AI, and some of its important concepts.\n\nArtificial Intelligence is a branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem-solving, understanding language, and recognizing images.'), Document(metadata={'source': 'speech.txt'}, page_content='One of the most important concepts in AI is Machine Learning. Machine Learning is a technique that allows computers to learn from data without being explicitly programmed for every task. For example, email spam filters use machine learning to identify unwanted emails based on previously observed patterns.'), Document(metadata={'source': 'speech.txt'}, page_content='To conclude, Artificial Intelligenc

## 3. Ollama Embeddings (local and free)

[Ollama](https://ollama.com) runs open-source models **on your own machine**, with no API key and no cost.

Setup:
```bash
ollama pull nomic-embed-text   # download the embedding model
```
The Ollama app/server must be running (default `http://localhost:11434`).

> ⚠️ `langchain_community.embeddings.OllamaEmbeddings` (next cell) is **deprecated**. The cell after it uses the recommended `langchain_ollama` package.

In [32]:
## ollama embeddings
from langchain_community.embeddings import OllamaEmbeddings


Recommended import from the `langchain-ollama` package.
*(Note: this reuses the variable name `embeddings`, so from here on it refers to the Ollama model, not OpenAI.)*

In [37]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [38]:
embeddings

OllamaEmbeddings(model='nomic-embed-text', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

`embed_documents` embeds several texts at once and returns one vector per text. `nomic-embed-text` produces **768-dimensional** vectors.

In [39]:
r1 = embeddings.embed_documents(
    [
        "Alpha is the first letter of the Greek alphabet",
        "Beta is the second letter of the Greek alphabet",
    ]
)

In [45]:
r1[1]

[0.030127922,
 0.024384327,
 -0.15029658,
 -0.06205362,
 0.11510293,
 -0.0058680517,
 -0.022715548,
 0.00068473205,
 -0.01705006,
 -0.03704833,
 -0.027235411,
 0.11953261,
 0.060340483,
 0.043182712,
 0.0476115,
 -0.07308244,
 0.005260628,
 -0.04406829,
 -0.033641003,
 0.0039947857,
 0.005657018,
 -0.022852816,
 -0.037680496,
 -0.03489714,
 0.10380479,
 0.0930221,
 0.03838677,
 0.030054383,
 -0.018078657,
 -0.06314749,
 -0.019125037,
 0.0036081746,
 0.04777389,
 -0.027218718,
 0.032447934,
 0.0031669887,
 -0.01704981,
 0.038607113,
 0.025545305,
 -0.0072443574,
 -0.013639308,
 -0.015395491,
 -0.008154716,
 0.024744833,
 0.05034828,
 -0.051196568,
 -0.01049495,
 0.085601315,
 0.07119415,
 -0.023600955,
 0.07217702,
 -0.016476316,
 0.010544258,
 0.009119209,
 0.002724619,
 -0.0019438097,
 0.051220912,
 -0.040656555,
 0.010607976,
 -0.010354058,
 0.057101313,
 0.05719231,
 -0.043067433,
 0.07344905,
 0.05484062,
 -0.014911564,
 -0.024579806,
 0.04289142,
 -0.010142792,
 -0.038999904,
 0.0

In [43]:
print( len(r1[0]))

768


`embed_query` is used for the search question. Its vector would be compared against the document vectors above.

In [44]:
embeddings.embed_query("What is the second letter of greek alphabet")

[0.02079488,
 0.00070505374,
 -0.12896974,
 -0.07129483,
 0.07980657,
 -0.035238914,
 -0.048224572,
 0.00031142888,
 -0.01607927,
 -0.01394682,
 -0.05367693,
 0.12309355,
 0.059656292,
 0.010410592,
 0.031516872,
 -0.054513626,
 0.02729701,
 -0.03392966,
 -0.05049483,
 0.004214593,
 -0.037520982,
 -0.021362402,
 -0.013335885,
 -0.025968278,
 0.099134065,
 0.07229861,
 0.087413944,
 0.037742198,
 -0.02536154,
 -0.053608626,
 -0.04219192,
 -0.015030754,
 0.04988531,
 -0.06668191,
 0.011468364,
 0.015127533,
 -0.015440434,
 0.04930094,
 0.007626899,
 -0.0076561524,
 -0.023757532,
 -0.05353447,
 -0.005852059,
 0.02358965,
 0.04752405,
 -0.07079366,
 -0.0608964,
 0.071158335,
 0.022894267,
 -0.059666596,
 0.047424994,
 -0.00021580824,
 -0.007145134,
 0.014590503,
 -0.02329025,
 0.009363142,
 0.029376986,
 -0.024035178,
 0.04382756,
 0.008761569,
 0.027443152,
 0.078679174,
 -0.03847738,
 0.05982134,
 0.038728457,
 0.013859394,
 0.0040621334,
 0.06469048,
 0.0008512292,
 -0.045106687,
 0.014

## 4. Hugging Face Embeddings (local and free)

`HuggingFaceEmbeddings` runs **sentence-transformers** models locally. The model is downloaded the first time and cached in `~/.cache/huggingface`.

`HF_TOKEN` is optional for public models, but it avoids rate limits and is required for gated models.

In [1]:
## HUgging face embeddings

import os 
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')

`sentence-transformers/all-MiniLM-L6-v2` is small and fast and produces **384-dimensional** vectors. It's a popular starting point.

*(The `tqdm` / symlink warnings in the output are harmless on Windows.)*

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\yokes\anaconda3\envs\langchain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\yokes\anaconda3\envs\langchain\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yokes\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. 

Output: 2 vectors, each with 384 dimensions, plus the first 10 values of the first vector.

In [4]:
r1 = embeddings.embed_documents(
    [
        "Alpha is the first letter of the Greek alphabet",
        "Beta is the second letter of the Greek alphabet"
    ]
)

print(len(r1))
print(len(r1[0]))
print(r1[0][:10])

2
384
[0.004850307013839483, -0.059995073825120926, -0.05643906444311142, -0.008184429258108139, -0.03675984591245651, -0.012441505677998066, 0.07796331495046616, -0.022450270131230354, 0.010831926949322224, 0.055206626653671265]


## Summary

| Provider | Model | Dimensions | Cost | Runs |
|---|---|---|---|---|
| OpenAI | `text-embedding-3-small` | 1536 (configurable) | Paid API | Cloud |
| Ollama | `nomic-embed-text` | 768 | Free | Local |
| Hugging Face | `all-MiniLM-L6-v2` | 384 | Free | Local |

⚠️ **Always use the same embedding model** for indexing documents and for embedding queries. Vectors from different models are not comparable.

➡️ **Next:** `04-VectorStore-FAISS.ipynb`, which stores and searches vectors with FAISS.